In [1]:
import psycopg2
import pandas as pd
from sqlalchemy import create_engine,URL

In [2]:
# connect to DB
conn = psycopg2.connect(
  host='localhost',
  port=5432,
  dbname='postgres',
  user='postgres',
  password='postgres'
)
conn.set_session(autocommit=True)
cursor = conn.cursor()

In [3]:
engine = create_engine(
  URL.create(
    drivername='postgresql+psycopg2',
    host='localhost',
    port=5432,
    database='postgres',
    username='postgres',
    password='postgres'
  )
)

In [4]:
# read m3_total
cursor.execute(
  'select * from m3_total'
)
m3_total_df = pd.DataFrame(
  cursor.fetchall(),
  columns=[col.name for col in cursor.description]
)

In [5]:
m3_total_df

,sales_amt,base_dt,base_year,base_month,name,perimeter,area,linear_value,convex_value,circular_value,...,sales_afternoon_cnt,sales_evening_cnt,sales_night_cnt,sales_male_cnt,sales_female_cnt,sales_age20_cnt,sales_age30_cnt,sales_age40_cnt,sales_age50_cnt,sales_age60_cnt
0,7.848960e+09,2023-12-01,2023,12,오류동역,2197.0,173400.0,0.012669,0.806334,0.355680,...,198670.0,114175.0,47183.0,211249.0,196458.0,75942.0,95925.0,76135.0,76600.0,83105.0
1,6.782354e+09,2022-06-01,2022,6,오류동역,2197.0,173400.0,0.012669,0.806334,0.355680,...,186689.0,123848.0,50478.0,211409.0,195090.0,83556.0,97169.0,77514.0,75474.0,72785.0
2,7.494432e+09,2023-10-01,2023,10,오류동역,2197.0,173400.0,0.012669,0.806334,0.355680,...,199891.0,124079.0,52549.0,220358.0,206624.0,81778.0,101387.0,77156.0,82714.0,83947.0
3,4.748296e+09,2021-02-01,2021,2,오류동역,2197.0,173400.0,0.012669,0.806334,0.355680,...,137534.0,89506.0,24963.0,145022.0,132345.0,63639.0,65510.0,53877.0,47416.0,46926.0
4,6.968481e+09,2024-01-01,2024,1,오류동역,2197.0,173400.0,0.012669,0.806334,0.355680,...,185335.0,109185.0,43355.0,198325.0,181325.0,57188.0,91943.0,69706.0,73614.0,87199.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3451,5.133690e+11,2023-07-01,2023,7,여의도,8752.0,4399501.0,0.001989,0.995550,0.386898,...,4297234.0,1434890.0,208485.0,4736559.0,4188197.0,1166849.0,2498455.0,2445096.0,1683542.0,1130815.0
3452,5.869216e+11,2021-04-01,2021,4,여의도,8752.0,4399501.0,0.001989,0.995550,0.386898,...,4042567.0,1382162.0,259382.0,4292698.0,3775268.0,1222567.0,2270157.0,2303842.0,1447717.0,823683.0
3453,5.611785e+11,2022-01-01,2022,1,여의도,8752.0,4399501.0,0.001989,0.995550,0.386898,...,4129570.0,1281180.0,236574.0,4176811.0,3725156.0,971138.0,2141751.0,2352742.0,1524621.0,911716.0
3454,4.932664e+11,2023-01-01,2023,1,여의도,8752.0,4399501.0,0.001989,0.995550,0.386898,...,4105739.0,1197882.0,190742.0,4255276.0,3691163.0,906612.0,2218566.0,2232459.0,1545969.0,1042834.0


---
x 구성

In [6]:
# building_info	용도별 건물 개수, 연면적, 평균 연식
cursor.execute(
  f'''
  select
    mk.name,
    sum(case
      when main_use = '주거' then 1
      else 0
    end) bld_res_cnt,
    sum(case
      when main_use = '근생' then 1
      else 0
    end) bld_ngh_cnt,
    sum(case
      when main_use = '업무' then 1
      else 0
    end) bld_off_cnt,
    sum(case
      when main_use = '판매' then 1
      else 0
    end) bld_ret_cnt,
    sum(case
      when main_use = '기타' then 1
      else 0
    end) bld_etc_cnt,
    sum(case
      when main_use = '주거' then tot_area
      else 0
    end) bld_res_area,
    sum(case
      when main_use = '근생' then tot_area
      else 0
    end) bld_ngh_area,
    sum(case
      when main_use = '업무' then tot_area
      else 0
    end) bld_off_area,
    sum(case
      when main_use = '판매' then tot_area
      else 0
    end) bld_ret_area,
    sum(case
      when main_use = '기타' then tot_area
      else 0
    end) bld_etc_area,
    sum(case
      when main_use = '주거' then parklot_cnt
      else 0
    end) bld_res_parklot,
    sum(case
      when main_use = '근생' then parklot_cnt
      else 0
    end) bld_ngh_parklot,
    sum(case
      when main_use = '업무' then parklot_cnt
      else 0
    end) bld_off_parklot,
    sum(case
      when main_use = '판매' then parklot_cnt
      else 0
    end) bld_ret_parklot,
    sum(case
      when main_use = '기타' then parklot_cnt
      else 0
    end) bld_etc_parklot,
    sum(case
      when main_use = '주거' then bld_age
      else 0
    end) bld_res_age,
    sum(case
      when main_use = '근생' then bld_age
      else 0
    end) bld_ngh_age,
    sum(case
      when main_use = '업무' then bld_age
      else 0
    end) bld_off_age,
    sum(case
      when main_use = '판매' then bld_age
      else 0
    end) bld_ret_age,
    sum(case
      when main_use = '기타' then bld_age
      else 0
    end) bld_etc_age
  from (
    select
      cname name,
      geom_3857
    from market_polygon
  ) mk,
  (
    select
      bld.pnu,
      case
        when use_nm in ('단독주택','공동주택') then '주거'
        when use_nm in ('제1종근린생활시설','제2종근린생활시설') then '근생'
        when use_nm in ('업무시설','교육연구시설') then '업무'
        when use_nm in ('판매시설','판매및영업시설') then '판매'
        else '기타'
      end main_use,
      tot_area,
      parklot_cnt,
      case
        when complete_dt is null or replace(complete_dt,' ','') = '' then 50
        else 2024 - left(complete_dt,4)::numeric
      end bld_age,
      lot.geom_3857
    from building_info bld
    inner join lot_polygon lot
    on bld.pnu = lot.pnu
  ) bld
  where
    st_intersects(
      mk.geom_3857,
      bld.geom_3857
    )
  group by 1
  '''
)
mk_bld = pd.DataFrame(
  cursor.fetchall(),
  columns=[col.name for col in cursor.description]
)

In [7]:
# public_land_price	토지 개수, 평균 공시지가 (면적 가중치)
cursor.execute(
  f'''
  select
    mk.name,
    count(plp.pnu) lot_cnt,
    avg(plp.plp_amt) lot_plp
  from (
    select
      cname name,
      geom_3857
    from market_polygon
  ) mk,
  (
    select
      plp.pnu,
      plp.amount plp_amt,
      lot.geom_3857
    from (
      select
        distinct on (pnu)
        pnu,
        base_year,
        amount
      from public_land_price
      order by pnu, base_year desc
    ) plp
    inner join lot_polygon lot
    on plp.pnu = lot.pnu
  ) plp
  where
    st_intersects(
      mk.geom_3857,
      plp.geom_3857
    )
  group by 1
  '''
)
mk_plp = pd.DataFrame(
  cursor.fetchall(),
  columns=[col.name for col in cursor.description]
)

In [8]:
# subway_ent	500m 반경 내 지하철 개수
cursor.execute(
  f'''
  select
    mk.name,
    count(sub.*) subway_cnt
  from (
    select
      cname name,
      geom_3857
    from market_polygon
  ) mk,
  (
    select
      station_nm,
      st_collect(geom_3857) geom_3857
    from subway_ent
    group by 1
  ) sub
  where
    st_intersects(
      st_buffer(
        mk.geom_3857,
        500
      ),
      sub.geom_3857
    )
  group by 1
  '''
)
mk_subway = pd.DataFrame(
  cursor.fetchall(),
  columns=[col.name for col in cursor.description]
)

In [10]:
# walk_pop	500m 반경 유동인구
cursor.execute(
  f'''
  select
    mk.name,
    extract('year' from walk.base_dt) base_year,
    extract('month' from walk.base_dt) base_month,
    sum(walk.tot_cnt) walk_pop
  from (
    select
      cname name,
      geom_3857
    from market_polygon
  ) mk,
  walk_pop walk
  where
    st_intersects(
      st_buffer(
        mk.geom_3857,
        500
      ),
      walk.geom_3857
    )
  group by 1,2,3
  '''
)
mk_walk = pd.DataFrame(
  cursor.fetchall(),
  columns=[col.name for col in cursor.description]
)

In [40]:
mk_walk['base_year'] = mk_walk['base_year'].astype('string')
mk_walk['base_month'] = mk_walk['base_month'].astype('string')

In [11]:
# live_pop	1000m 반경 거주인구
cursor.execute(
  f'''
  select
    mk.name,
    sum(live.pop_cnt) live_pop
  from (
    select
      cname name,
      geom_3857
    from market_polygon
  ) mk,
  live_pop live
  where
    st_intersects(
      st_buffer(
        mk.geom_3857,
        1000
      ),
      live.geom_3857
    )
  group by 1
  '''
)
mk_live = pd.DataFrame(
  cursor.fetchall(),
  columns=[col.name for col in cursor.description]
)

In [12]:
# work_pop	500m 반경 직장인구
cursor.execute(
  f'''
  select
    mk.name,
    sum(work.pop_cnt) work_pop
  from (
    select
      cname name,
      geom_3857
    from market_polygon
  ) mk,
  work_pop work
  where
    st_intersects(
      st_buffer(
        mk.geom_3857,
        500
      ),
      work.geom_3857
    )
  group by 1
  ;
  '''
)
mk_work = pd.DataFrame(
  cursor.fetchall(),
  columns=[col.name for col in cursor.description]
)

---
m5_y + x 통합

In [41]:
m5_total_df = m3_total_df.merge(
  mk_bld,
  how='left',
  on='name'
).merge(
  mk_plp,
  how='left',
  on='name'
).merge(
  mk_subway,
  how='left',
  on='name'
).merge(
  mk_walk,
  how='left',
  on=['base_year','base_month','name']
).merge(
  mk_live,
  how='left',
  on='name'
).merge(
  mk_work,
  how='left',
  on='name'
)

In [45]:
m5_total_df.to_sql(
  'm5_total',
  engine,
  if_exists='replace',
  index=False
)

132

---
전달용 데이터 저장하기

In [46]:
m5_total_df.to_csv(
  'm5_data_06.24.csv',
  sep=',',
  index=False
)